# House Prices - Fase 3: Modelo Baseline

Vamos treinar os primeiros modelos de Machine Learning!

**O que vamos fazer:**
1. Carregar dados limpos da Fase 2
2. Ridge Regression (baseline linear)
3. Random Forest (modelo de arvore)
4. Comparar e escolher o melhor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

sns.set_style('whitegrid')
DATA_DIR = '../data'
REPORTS_DIR = '../reports'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)
print('OK')

In [ ]:
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train_clean.csv'))
y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train_clean.csv')).squeeze()
X_test = pd.read_csv(os.path.join(DATA_DIR, 'X_test_clean.csv'))
test_original = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

def evaluate_rmse(model, X, y, cv=5):
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
    return np.sqrt(-scores)

print('Funcoes definidas')

## Modelo 1: Ridge Regression

Regressao linear com regularizacao L2. Serve como baseline.

A regularizacao evita coeficientes muito grandes (overfitting).

In [ ]:
ridge = Ridge(alpha=1.0, random_state=42)
ridge_scores = evaluate_rmse(ridge, X_train, y_train)
print(f'Ridge RMSE: {ridge_scores.mean():.4f} (+/- {ridge_scores.std():.4f})')

best_alpha = None
best_score = float('inf')
for alpha in [0.1, 1.0, 10.0, 100.0]:
    model = Ridge(alpha=alpha, random_state=42)
    scores = evaluate_rmse(model, X_train, y_train)
    print(f'  alpha={alpha}: {scores.mean():.4f}')
    if scores.mean() < best_score:
        best_score = scores.mean()
        best_alpha = alpha
print(f'Melhor alpha: {best_alpha}')

## Modelo 2: Random Forest

Modelo baseado em arvores de decisao. Captura relacoes nao-lineares.

Cada arvore 'vota' na previsao final -> mais robusto que uma unica arvore.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_scores = evaluate_rmse(rf, X_train, y_train)
print(f'Random Forest RMSE: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})')

In [ ]:
ridge_best = Ridge(alpha=best_alpha, random_state=42)
ridge_best.fit(X_train, y_train)
rf.fit(X_train, y_train)

ridge_pred = ridge_best.predict(X_train)
rf_pred = rf.predict(X_train)

ridge_rmsle = rmsle(y_train, ridge_pred)
rf_rmsle = rmsle(y_train, rf_pred)

print('Comparacao (dados de treino):')
print(f'  Ridge:         RMSE log={best_score:.4f}  RMSLE={ridge_rmsle:.4f}')
print(f'  Random Forest: RMSE log={rf_scores.mean():.4f}  RMSLE={rf_rmsle:.4f}')

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 8))
top15.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 Features - Random Forest')
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', dpi=150)
plt.close()

print('Top 10 features:')
for feat, imp in top15.head(10).items():
    print(f'  {feat:<25} {imp:.4f}')

In [ ]:
best_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)
y_pred_log = best_model.predict(X_test)
y_pred_real = np.expm1(y_pred_log)
submission = pd.DataFrame({'Id': test_original['Id'], 'SalePrice': y_pred_real})
submission.to_csv('../data/submission.csv', index=False)
print('Submission gerada:', len(submission), 'previsoes')
media_price = submission['SalePrice'].mean()
print('Media:', round(media_price))
print(submission.head())


## Resultados e Proximos Passos

**Fase 3 concluida!** Treinamos dois modelos:
- Ridge: modelo linear simples
- Random Forest: modelo baseado em arvores

**Proximos passos:**
1. Tuning de hiperparametros (GridSearch)
2. XGBoost (modelo avancado)
3. Ensemble (combinar modelos)
4. Submissao para o Kaggle!